# Упражнения по регуляризации и методу градиентного спуска
![UnderOverFit.png](Assets/UnderOverFit.png)


# Учебные цели

- Объяснить суть функций затрат, регуляризации, выбора признаков и гиперпараметров
- Охарактеризовать сложные алгоритмы статистической оптимизации, такие как метод градиентного спуска, и его применение в линейной регрессии
- Применять расширение Intel® для Scikit-learn*, чтобы использовать вычислительные возможности аппаратного обеспечения


## Введение

Мы начнём с краткого введения в регрессию, полиномиальные признаки и регуляризацию на примере очень простого набора разреженных данных, содержащего столбец данных `x` и соответствующие данные `y` с шумом. Файл данных называется `X_Y_Sinusoid_Data.csv`.

In [ ]:
from __future__ import print_function
import os
data_path = ['data']

from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import MinMaxScaler

from sklearnex import patch_sklearn
patch_sklearn()

## Вопрос 1

* Импортируйте данные.

* Также сгенерируйте примерно 100 равномерно распределенных точек данных по оси x в диапазоне от 0 до 1. Используя эти точки, вычислите данные по оси y, представляющие «истинное значение» (реальную функцию), по уравнению: $y = \sin(2\pi x)$

* Постройте график разреженных данных (`x` против `y`) и вычисленных («реальных») данных.



In [ ]:
import pandas as pd
import numpy as np

filepath = os.sep.join(data_path + ['X_Y_Sinusoid_Data.csv'])
data = pd.read_csv(filepath)

X_real = np.linspace(0, 1.0, 100)
Y_real = np.sin(2 * np.pi * X_real)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
sns.set_style('white')
sns.set_context('talk')
sns.set_palette('dark')

# Plot of the noisy (sparse)
ax = data.set_index('x')['y'].plot(ls='', marker='o', label='data')
ax.plot(X_real, Y_real, ls='--', marker='', label='real function')

ax.legend()
ax.set(xlabel='x data', ylabel='y data');


## Вопрос 2


* Используя класс `PolynomialFeatures` из библиотеки предварительной обработки данных Scikit-learn, создайте полиномиальные признаки 20-го порядка.
* Проведите линейную регрессию по этим данным.
* Постройте график, на котором будут отображены полученные прогнозируемые значения в сравнении с расчётными данными.

Обратите внимание, что для класса `PolynomialFeatures` требуется либо датафрейм (с одним столбцом, а не серия), либо двумерный массив размером (`X`, 1), где `X` — длина.

In [ ]:
# Setup the polynomial features
degree = 20
pf = PolynomialFeatures(degree)
lr = LinearRegression()

# Extract the X- and Y- data from the dataframe
X_data = data[['x']].to_numpy()
Y_data = data['y'].to_numpy()

# Create the features and fit the model
X_poly = pf.fit_transform(X_data)
lr = lr.fit(X_poly, Y_data)
Y_pred = lr.predict(X_poly)

# Plot the result
plt.plot(X_data, Y_data, marker='o', ls='', label='data', alpha=1)
plt.plot(X_real, Y_real, ls='--', label='real function')
plt.plot(X_data, Y_pred, marker='^', alpha=.5, label='predictions w/ polynomial features')
plt.legend()
ax = plt.gca()
ax.set(xlabel='x data', ylabel='y data');


## Вопрос 3

* Проведите регрессионный анализ данных с полиномиальными признаками с использованием регрессии Ridge ($\alpha$ = 0,001) и регрессии Lasso ($\alpha$ = 0,0001).
* Постройте графики результатов, как это было сделано в вопросе 1.
* Также постройте график величин коэффициентов, полученных в результате этих регрессий, и сравните их с коэффициентами, полученными при линейной регрессии в предыдущем вопросе. Коэффициенты линейной регрессии, вероятно, потребуют отдельного графика (или собственной оси y) из-за их больших значений.

Что говорит вам сравнительно большая величина коэффициентов о роли регуляризации?

In [ ]:
# Mute the sklearn warning about regularization
import warnings
warnings.filterwarnings('ignore', module='sklearn')

# The ridge regression model
rr = Ridge(alpha=0.001)
rr = rr.fit(X_poly, Y_data)
Y_pred_rr = rr.predict(X_poly)

# The lasso regression model
lassor = Lasso(alpha=0.0001)
lassor = lassor.fit(X_poly, Y_data)
Y_pred_lr = lassor.predict(X_poly)

# The plot of the predicted values
plt.plot(X_data, Y_data, marker='o', ls='', label='data')
plt.plot(X_real, Y_real, ls='--', label='real function')
plt.plot(X_data, Y_pred, label='linear regression', marker='^', alpha=.5)
plt.plot(X_data, Y_pred_rr, label='ridge regression', marker='^', alpha=.5)
plt.plot(X_data, Y_pred_lr, label='lasso regression', marker='^', alpha=.5)

plt.legend()

ax = plt.gca()
ax.set(xlabel='x data', ylabel='y data');

In [ ]:
# let's look at the absolute value of coefficients for each model

coefficients = pd.DataFrame()
coefficients['linear regression'] = lr.coef_.ravel()
coefficients['ridge regression'] = rr.coef_.ravel()
coefficients['lasso regression'] = lassor.coef_.ravel()
coefficients = coefficients.applymap(abs)

coefficients.describe()  # Huge difference in scale between non-regularized vs regularized regression

In [ ]:
colors = sns.color_palette()

# Setup the dual y-axes
ax1 = plt.axes()
ax2 = ax1.twinx()

# Plot the linear regression data
ax1.plot(lr.coef_.ravel(),
         color=colors[0], marker='o', label='linear regression')

# Plot the regularization data sets
ax2.plot(rr.coef_.ravel(),
         color=colors[1], marker='o', label='ridge regression')

ax2.plot(lassor.coef_.ravel(),
         color=colors[2], marker='o', label='lasso regression')

# Customize axes scales
ax1.set_ylim(-2e14, 2e14)
ax2.set_ylim(-25, 25)

# Combine the legends
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1+h2, l1+l2)

ax1.set(xlabel='coefficients',ylabel='linear regression')
ax2.set(ylabel='ridge and lasso regression')

ax1.set_xticks(range(len(lr.coef_)));

## Вопрос 4

В оставшихся вопросах мы будем работать с [набором данных](https://www.kaggle.com/c/house-prices-advanced-regression-techniques) из прошлого занятия, который основан на данных о ценах на жилье в Эймсе, штат Айова. В наборе содержится большое количество признаков — подробнее об этих признаках см. в упражнениях третьей недели.

Для начала:

* Импортируйте данные с помощью Pandas, удалите все нулевые значения и выполните кодирование категориальных переменных методом «one-hot». Для этого можно использовать либо кодировщики признаков из Scikit-learn, либо метод `get_dummies` из Pandas.
* Разделите данные на обучающую и тестовую выборки.
* Примените логарифмическое преобразование к асимметричным признакам.
* Можно попробовать выполнить масштабирование, хотя будет интересно посмотреть, насколько хорошо работает регуляризация без масштабирования признаков.

In [ ]:
filepath = os.sep.join(data_path + ['Ames_Housing_Sales.csv'])
data = pd.read_csv(filepath, sep=',')

Создайте список категориальных данных и примените к ним кодирование «one-hot». Кодировщик «one-hot» в Pandas (`get_dummies`) хорошо подходит для данных, определённых как категориальные.

In [ ]:
# Get a Pd.Series consisting of all the string categoricals
one_hot_encode_cols = data.dtypes[data.dtypes == object]  # filtering by string categoricals
one_hot_encode_cols = one_hot_encode_cols.index.tolist()  # list of categorical fields

# Here we see another way of one-hot-encoding:
# Encode these columns as categoricals so one hot encoding works on split data (if desired)
for col in one_hot_encode_cols:
    data[col] = pd.Categorical(data[col])

# Do the one hot encoding
data = pd.get_dummies(data, columns=one_hot_encode_cols)

Далее разделите данные на обучающий и тестовый наборы.

In [ ]:
train, test = train_test_split(data, test_size=0.3, random_state=42)

Есть ряд столбцов с асимметричным распределением — к ним можно применить логарифмическое преобразование. Обратите внимание, что в их число входит и `SalePrice` — наш предиктор. Однако давайте оставим его без изменений.

In [ ]:
# Create a list of float colums to check for skewing
mask = data.dtypes == float
float_cols = data.columns[mask]

In [ ]:
skew_limit = 0.75
skew_vals = train[float_cols].skew()

skew_cols = (skew_vals
             .sort_values(ascending=False)
             .to_frame()
             .rename(columns={0:'Skew'})
             .query('abs(Skew) > {0}'.format(skew_limit)))

skew_cols

Преобразуйте все столбцы, в которых коэффициент асимметрии превышает 0,75, за исключением столбца «SalePrice».

In [ ]:
# OPTIONAL: Let's look at what happens to one of these features, when we apply np.log1p visually.

field = "BsmtFinSF1"
fig, (ax_before, ax_after) = plt.subplots(1, 2, figsize=(10, 5))
train[field].hist(ax=ax_before)
train[field].apply(np.log1p).hist(ax=ax_after)
ax_before.set(title='before np.log1p', ylabel='frequency', xlabel='value')
ax_after.set(title='after np.log1p', ylabel='frequency', xlabel='value')
fig.suptitle('Field "{}"'.format(field));
# a little bit better

In [ ]:
# Mute the setting wtih a copy warnings
pd.options.mode.chained_assignment = None

for col in skew_cols.index.tolist():
    if col == "SalePrice":
        continue
    train[col] = np.log1p(train[col])
    test[col]  = test[col].apply(np.log1p)  # same thing

Выделить признаки из набора предикторов.

In [ ]:
feature_cols = [x for x in train.columns if x != 'SalePrice']
X_train = train[feature_cols].to_numpy()
y_train = train['SalePrice'].to_numpy()

X_test  = test[feature_cols].to_numpy()
y_test  = test['SalePrice'].to_numpy()

## Вопрос 5

* Напишите функцию **`rmse`**, которая принимает в качестве аргументов значения истинных и прогнозируемых значений и возвращает среднеквадратичную ошибку. Используйте функцию `mean_squared_error` из библиотеки sklearn.


In [ ]:
def rmse(ytrue, ypredicted):
    return np.sqrt(mean_squared_error(ytrue, ypredicted))

* Построить базовую модель линейной регрессии
* Вывести среднеквадратичную ошибку для этой модели
* Построить график сопоставления прогнозируемой и фактической цены продажи на основе модели.

In [ ]:
linearRegression = LinearRegression().fit(X_train, y_train)

linearRegression_rmse = rmse(y_test, linearRegression.predict(X_test))

print(linearRegression_rmse)

In [ ]:
f = plt.figure(figsize=(6,6))
ax = plt.axes()

ax.plot(y_test, linearRegression.predict(X_test),
         marker='o', ls='', ms=3.0)

lim = (0, y_test.max())

ax.set(xlabel='Actual Price',
       ylabel='Predicted Price',
       xlim=lim,
       ylim=lim,
       title='Linear Regression Results');

## Вопрос 6

В регрессии Риджа используется L2-нормализация для уменьшения величины коэффициентов. Это может быть полезно в ситуациях с высокой дисперсией. Каждая из функций регуляризации в Scikit-learn содержит версии со встроенной перекрестной валидацией.

* Постройте обычную (без перекрестной валидации) модель Риджа для диапазона значений $\alpha$ и постройте график среднеквадратичной ошибки (RMSE) с использованием созданной вами выше функции ошибки с перекрестной валидацией.
* Используйте $$[0,005; 0,05; 0,1; 0,3; 1; 3; 5; 10; 15; 30; 80]$$ в качестве диапазона значений $\alpha$.
* Затем повторите подбор моделей Риджа, используя диапазон значений $\alpha$ из предыдущего раздела. Сравните результаты.

Теперь рассмотрим метод `RidgeCV`. К сожалению, получить значения альфа для моделей, которые не были выбраны, не представляется возможным. Полученные значения ошибок и значения $\alpha$ очень похожи на те, что были получены выше.

In [ ]:
alphas = [0.005, 0.05, 0.1, 0.3, 1, 3, 5, 10, 15, 30, 80]

ridgeCV = RidgeCV(alphas=alphas,
                  cv=4).fit(X_train, y_train)

ridgeCV_rmse = rmse(y_test, ridgeCV.predict(X_test))

print(ridgeCV.alpha_, ridgeCV_rmse)

## Вопрос 7


Подобно функции `RidgeCV`, существует также функция `LassoCV`, которая использует функцию регуляризации L1 и перекрестную валидацию. Регуляризация L1 выборочно уменьшает некоторые коэффициенты, что фактически приводит к удалению признаков.

Функция `LassoCV` не позволяет задавать функцию оценки. Однако для оценки ошибки конечной модели можно использовать созданную выше пользовательскую функцию ошибки (`rmse`).

Аналогично существует также функция ElasticNet с перекрестной валидацией — `ElasticNetCV`, представляющая собой комбинацию регуляризаций L2 и L1.

* Постройте модель Lasso с использованием перекрестной валидации и определите оптимальное значение $\alpha$ и RMSE с помощью созданной выше функции. Обратите внимание, что величина $\alpha$ может отличаться от значения для модели Ridge.
* Повторите эту процедуру для модели ElasticNet.
* Сравните результаты с помощью таблицы и/или графика.

Используйте следующие значения $\alpha$:
`[1e-5, 5e-5, 0.0001, 0.0005]`

In [ ]:
alphas2 = np.array([1e-5, 5e-5, 0.0001, 0.0005])

lassoCV = LassoCV(alphas=alphas2,
                  max_iter=5e4,
                  cv=3).fit(X_train, y_train)

lassoCV_rmse = rmse(y_test, lassoCV.predict(X_test))

print(lassoCV.alpha_, lassoCV_rmse)  # Lasso is slower

Мы можем определить, сколько из этих характеристик остаются неравными нулю.

In [ ]:
print('Of {} coefficients, {} are non-zero with Lasso.'.format(len(lassoCV.coef_),
                                                               len(lassoCV.coef_.nonzero()[0])))

Теперь попробуйте использовать метод «Elastic Net» с теми же значениями альфа, что и в методе «Lasso», и значениями l1_ratios в диапазоне от 0,1 до 0,9

In [ ]:
l1_ratios = np.linspace(0.1, 0.9, 9)

elasticNetCV = ElasticNetCV(alphas=alphas2,
                            l1_ratio=l1_ratios,
                            max_iter=1e4).fit(X_train, y_train)
elasticNetCV_rmse = rmse(y_test, elasticNetCV.predict(X_test))

print(elasticNetCV.alpha_, elasticNetCV.l1_ratio_, elasticNetCV_rmse)

Сравнить результаты расчета среднеквадратичной ошибки (RMSE) для всех моделей проще всего в таблице.

In [ ]:
rmse_vals = [linearRegression_rmse, ridgeCV_rmse, lassoCV_rmse, elasticNetCV_rmse]

labels = ['Linear', 'Ridge', 'Lasso', 'ElasticNet']

rmse_df = pd.Series(rmse_vals, index=labels).to_frame()
rmse_df.rename(columns={0: 'RMSE'}, inplace=1)
rmse_df

Мы также можем построить график, на котором будут отображены фактические и прогнозируемые цены на жилье, как и ранее.

In [ ]:
f = plt.figure(figsize=(6,6))
ax = plt.axes()

labels = ['Ridge', 'Lasso', 'ElasticNet']

models = [ridgeCV, lassoCV, elasticNetCV]

for mod, lab in zip(models, labels):
    ax.plot(y_test, mod.predict(X_test),
             marker='o', ls='', ms=3.0, label=lab)


leg = plt.legend(frameon=True)
leg.get_frame().set_edgecolor('black')
leg.get_frame().set_linewidth(1.0)

ax.set(xlabel='Actual Price',
       ylabel='Predicted Price',
       title='Linear Regression Results');

## Вопрос 8

В этом упражнении рассмотрим метод стохастического градиентного спуска (SGD).
Напомним, что линейные модели в целом чувствительны к масштабированию.
Однако метод SGD *очень* чувствителен к масштабированию.
Кроме того, высокое значение скорости обучения может привести к расхождению алгоритма, тогда как слишком низкое значение может привести к слишком длительному процессу сходимости.

* Постройте модель со стохастическим градиентным спуском без штрафа регуляризации (соответствующий параметр — `penalty`).
* Теперь постройте модели со стохастическим градиентным спуском с каждым из трёх штрафов (L2, L1, Elastic Net), используя значения параметров, определённые выше с помощью перекрёстной валидации.
* Не масштабируйте данные перед построением модели.
* Сравните результаты с теми, которые были получены без использования метода стохастического градиентного спуска.

In [ ]:
# Import SGDRegressor and prepare the parameters
model_parameters_dict = {
    'Linear': {'penalty': 'none'},
    'Lasso': {'penalty': 'l2',
           'alpha': lassoCV.alpha_},
    'Ridge': {'penalty': 'l1',
           'alpha': ridgeCV_rmse},
    'ElasticNet': {'penalty': 'elasticnet',
                   'alpha': elasticNetCV.alpha_,
                   'l1_ratio': elasticNetCV.l1_ratio_}
}

new_rmses = {}
for modellabel, parameters in model_parameters_dict.items():
    # following notation passes the dict items as arguments
    SGD = SGDRegressor(**parameters)
    SGD.fit(X_train, y_train)
    new_rmses[modellabel] = rmse(y_test, SGD.predict(X_test))

rmse_df['RMSE-SGD'] = pd.Series(new_rmses)
rmse_df

Обратите внимание, насколько велики значения ошибок! Алгоритм расходится. Это может быть связано со слишком высоким коэффициентом масштабирования и/или слишком высокой скоростью обучения. Давайте скорректируем скорость обучения и посмотрим, что произойдет.

In [ ]:
# Import SGDRegressor and prepare the parameters
model_parameters_dict = {
    'Linear': {'penalty': 'none'},
    'Lasso': {'penalty': 'l2',
           'alpha': lassoCV.alpha_},
    'Ridge': {'penalty': 'l1',
           'alpha': ridgeCV_rmse},
    'ElasticNet': {'penalty': 'elasticnet',
                   'alpha': elasticNetCV.alpha_,
                   'l1_ratio': elasticNetCV.l1_ratio_}
}

new_rmses = {}
for modellabel, parameters in model_parameters_dict.items():
    # following notation passes the dict items as arguments
    SGD = SGDRegressor(eta0=1e-7, **parameters)
    SGD.fit(X_train, y_train)
    new_rmses[modellabel] = rmse(y_test, SGD.predict(X_test))

rmse_df['RMSE-SGD-learningrate'] = pd.Series(new_rmses)
rmse_df

Now let's scale our training data and try again.

* Fit a `MinMaxScaler` to `X_train` create a variable `X_train_scaled`.
* Using the scaler, transform `X_test` and create a variable `X_test_scaled`.
* Apply the same versions of SGD to them and compare the results. Don't pass in a eta0 this time.

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

new_rmses = {}
for modellabel, parameters in model_parameters_dict.items():
    # following notation passes the dict items as arguments
    SGD = SGDRegressor(**parameters)
    SGD.fit(X_train_scaled, y_train)
    new_rmses[modellabel] = rmse(y_test, SGD.predict(X_test_scaled))

rmse_df['RMSE-SGD-scaled'] = pd.Series(new_rmses)
rmse_df

In [ ]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

new_rmses = {}
for modellabel, parameters in model_parameters_dict.items():
    # following notation passes the dict items as arguments
    SGD = SGDRegressor(**parameters)
    SGD.fit(X_train_scaled, y_train)
    new_rmses[modellabel] = rmse(y_test, SGD.predict(X_test_scaled))

rmse_df['RMSE-SGD-scaled'] = pd.Series(new_rmses)
rmse_df